# Docquity Doctor Search Assessment

This notebook covers Task 1 intent classification and Task 2 query-to-content retrieval for the Senior Data Scientist NLP technical assessment.

## Phase 0: Setup

Project-wide configuration lives in `src/config.py` so notebook runs, scripts, and exported outputs share the same paths, random seed, split settings, BM25 parameters, and evaluation cutoffs.

In [ ]:
from src.config import CONFIG, set_random_seed
from src.data import load_all_raw

set_random_seed(CONFIG.random_seed)
raw = load_all_raw()
{name: df.shape for name, df in raw.items()}

## Phase 1: Data Audit

Audit raw queries, content metadata, behavioral signals, and impressions before defining labels or retrieval experiments.

### Phase 1 pipeline runners

These notebook-owned functions orchestrate the audit, schema design, and contextual-slot extraction. Their reusable helper logic remains in `src/`.

In [ ]:
# Phase-level orchestration is kept in this main notebook for reviewer visibility.
# Detailed, reusable helpers remain in src/.

import json
from pathlib import Path
import pandas as pd

from src.audit import (
    AUDIT_DIR,
    AuditPaths,
    audit_behavioral_signals,
    audit_content,
    audit_impressions,
    audit_queries,
    write_report as write_audit_report,
)
from src.config import PROCESSED_DATA_DIR, set_random_seed
from src.data import load_all_raw, load_content, load_queries
from src.query_schema import (
    SCHEMA_DIR,
    build_candidate_profile,
    build_schema_table,
    write_schema_report,
)
from src.query_extraction import (
    ENRICHED_QUERIES_PATH,
    EXTRACTION_EXAMPLES_PATH,
    EXTRACTION_QA_PATH,
    EXTRACTION_REPORT_PATH,
    EXTRACTION_SUMMARY_PATH,
    PHASE16_DIR,
    build_examples,
    build_quality_review_sample,
    build_summary,
    enrich_queries,
    write_report as write_extraction_report,
)

def run_phase1_audit() -> AuditPaths:
    set_random_seed()
    AUDIT_DIR.mkdir(parents=True, exist_ok=True)
    paths = AuditPaths(
        report=AUDIT_DIR / "phase1_data_audit.md",
        metrics=AUDIT_DIR / "phase1_metrics.json",
        query_manual_review=AUDIT_DIR / "phase1_query_manual_review.csv",
        query_near_duplicates=AUDIT_DIR / "phase1_query_near_duplicates.csv",
        content_near_duplicates=AUDIT_DIR / "phase1_content_near_duplicates.csv",
        suspicious_behavior=AUDIT_DIR / "phase1_suspicious_behavior.csv",
        table_dir=AUDIT_DIR / "tables",
    )
    data = load_all_raw()
    metrics = {
        "queries": audit_queries(data["queries"], paths),
        "content": audit_content(data["content"], paths),
        "behavioral_signals": audit_behavioral_signals(data["behavioral_signals"], paths),
        "impressions": audit_impressions(data["impressions"], data["behavioral_signals"], paths),
    }
    paths.metrics.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    write_audit_report(paths, metrics)
    return paths


def run_phase15_schema() -> dict[str, Path]:
    queries = load_queries()
    content = load_content()
    SCHEMA_DIR.mkdir(parents=True, exist_ok=True)

    candidate_profile = build_candidate_profile(queries, content)
    schema_table = build_schema_table()

    candidate_path = SCHEMA_DIR / "phase15_candidate_slot_profile.csv"
    schema_path = SCHEMA_DIR / "phase15_extended_query_schema.csv"
    report_path = SCHEMA_DIR / "phase15_extended_query_schema.md"

    candidate_profile.to_csv(candidate_path, index=False)
    schema_table.to_csv(schema_path, index=False)
    write_schema_report(report_path, candidate_profile, schema_table)

    return {
        "report": report_path,
        "schema": schema_path,
        "candidate_profile": candidate_path,
    }


def run_phase16_extraction() -> dict[str, Path]:
    set_random_seed()
    PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
    PHASE16_DIR.mkdir(parents=True, exist_ok=True)

    queries = load_queries()
    enriched = enrich_queries(queries)
    summary = build_summary(enriched)
    slot_columns = summary["column"].tolist()
    any_slot_mask = pd.Series(False, index=enriched.index)
    for column in slot_columns:
        if enriched[column].dtype == bool:
            any_slot_mask |= enriched[column]
        else:
            any_slot_mask |= enriched[column].notna()
    summary.attrs["queries_with_any_slot"] = int(any_slot_mask.sum())
    examples = build_examples(enriched)
    qa_sample = build_quality_review_sample(enriched)

    enriched.to_csv(ENRICHED_QUERIES_PATH, index=False)
    summary.to_csv(EXTRACTION_SUMMARY_PATH, index=False)
    examples.to_csv(EXTRACTION_EXAMPLES_PATH, index=False)
    qa_sample.to_csv(EXTRACTION_QA_PATH, index=False)
    write_extraction_report(EXTRACTION_REPORT_PATH, ENRICHED_QUERIES_PATH, summary, examples, qa_sample)

    return {
        "enriched_queries": ENRICHED_QUERIES_PATH,
        "summary": EXTRACTION_SUMMARY_PATH,
        "examples": EXTRACTION_EXAMPLES_PATH,
        "qa_sample": EXTRACTION_QA_PATH,
        "report": EXTRACTION_REPORT_PATH,
    }

In [ ]:
phase1_audit_paths = run_phase1_audit()
phase15_schema_paths = run_phase15_schema()
phase16_extraction_paths = run_phase16_extraction()

{
    "phase1_report": phase1_audit_paths.report,
    "phase15_report": phase15_schema_paths["report"],
    "phase16_report": phase16_extraction_paths["report"],
}

## Phase 2: Literature-Seeded Intent Taxonomy Refinement

The literature taxonomy is a seed rather than a fixed label set. The analysis combines delexicalized word unigram/bigram TF-IDF, cached frozen multilingual MiniLM embeddings, 13 structured NER/context features, prototype similarity and ambiguity margins, independent clustering, and manual clinical-coherence review. Final support is estimated with an explicit CPU adaptation of Liang et al.'s three-branch model: frozen semantic similarity, 200-feature sentence TF-IDF, and class-centroid TF-IDF are projected into a shared space, combined by learned sample-wise attention, and passed through a nonlinear classification head. Because Phase 2 has no gold labels, ordered boundary rules provide weak pseudo-labels where Liang's supervised model uses gold labels; reported validation measures weak-label agreement, not clinical accuracy.

Primary sources: [Monsalve et al. (2026)](https://doi.org/10.64898/2026.06.26.26356340) and [Liang et al. (2025)](https://doi.org/10.1038/s41598-025-25783-x).

### Phase 2 pipeline runner

This notebook-owned function exposes every Phase 2 substep in the main deliverable while delegating reusable algorithms to `src/taxonomy.py` and `src/liang_intent.py`.

In [ ]:
# Phase 2 orchestration is defined here; reusable modeling helpers remain in src/taxonomy.py
# and src/liang_intent.py.

from dataclasses import asdict
import re

import numpy as np
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.config import CACHE_DIR, CONFIG, FIGURES_DIR, PROCESSED_DATA_DIR, TAXONOMY_DIR
from src.taxonomy import (
    CANDIDATE_INTENTS,
    FEATURE_COLUMNS,
    INTENT_STOP_WORDS,
    SEED_INTENTS,
    _fit_cluster_models,
    _write_completion_checklist,
    _write_figures,
    _write_report,
    build_analysis_tables,
    build_manual_review,
    build_seed_to_final_derivation,
    build_structured_features,
    build_taxonomy_decisions,
    delexicalize_query,
    encode_texts,
    weak_anchor_assignment,
)
from src.liang_intent import (
    LiangAdaptationConfig,
    candidate_document,
    fit_liang_hybrid_intent_model,
)

def run_phase2_taxonomy(force_recompute_embeddings: bool = False) -> dict[str, Path]:
    """Run every Phase 2 substep and write its auditable artifacts."""
    set_random_seed()
    TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    queries_path = PROCESSED_DATA_DIR / "queries_with_contextual_slots.csv"
    queries = pd.read_csv(queries_path)
    queries["intent_text"] = queries.apply(delexicalize_query, axis=1)
    # Apply intent cues after entity replacement. Otherwise disease names such
    # as "Heart Failure" and "Hypertension in Pregnancy" leak the words
    # "failure" and "pregnancy" into the primary-intent estimate.
    anchors = queries["intent_text"].map(weak_anchor_assignment)
    queries[["anchor_rule_intent", "matched_candidate_intents", "n_anchor_matches"]] = pd.DataFrame(
        anchors.tolist(), index=queries.index
    )

    structured = build_structured_features(queries)
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        token_pattern=r"(?u)\b\w[\w+-]+\b",
        stop_words=list(INTENT_STOP_WORDS),
    )
    tfidf = vectorizer.fit_transform(queries["intent_text"])
    save_npz(CACHE_DIR / "phase2_tfidf_matrix.npz", tfidf)
    (CACHE_DIR / "phase2_tfidf_vocabulary.json").write_text(
        json.dumps({term: int(index) for term, index in vectorizer.vocabulary_.items()}, indent=2, sort_keys=True),
        encoding="utf-8",
    )

    prototype_texts = [intent.definition for intent in SEED_INTENTS]
    query_embeddings = encode_texts(
        queries["intent_text"].tolist(),
        CACHE_DIR / "phase2_query_embeddings.npz",
        force_recompute=force_recompute_embeddings,
    )
    prototype_embeddings = encode_texts(
        prototype_texts,
        CACHE_DIR / "phase2_prototype_embeddings.npz",
        force_recompute=force_recompute_embeddings,
    )
    candidate_documents = [candidate_document(intent) for intent in CANDIDATE_INTENTS]
    candidate_embeddings = encode_texts(
        candidate_documents,
        CACHE_DIR / "phase2_candidate_intent_embeddings.npz",
        force_recompute=force_recompute_embeddings,
    )
    prototype_scores = cosine_similarity(query_embeddings, prototype_embeddings)
    seed_names = [intent.literature_intent for intent in SEED_INTENTS]
    order = np.argsort(prototype_scores, axis=1)[:, ::-1]
    prototype_assignments = pd.DataFrame(
        {
            "query_id": queries["query_id"],
            "prototype_top_intent": [seed_names[i] for i in order[:, 0]],
            "prototype_top_score": prototype_scores[np.arange(len(queries)), order[:, 0]],
            "prototype_second_intent": [seed_names[i] for i in order[:, 1]],
            "prototype_second_score": prototype_scores[np.arange(len(queries)), order[:, 1]],
        }
    )
    prototype_assignments["ambiguity_margin"] = (
        prototype_assignments["prototype_top_score"] - prototype_assignments["prototype_second_score"]
    )
    prototype_assignments["margin_is_calibrated_confidence"] = False
    for i, name in enumerate(seed_names):
        safe_name = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        prototype_assignments[f"similarity_{safe_name}"] = prototype_scores[:, i].round(6)
    queries["prototype_top_intent"] = prototype_assignments["prototype_top_intent"]

    # Liang et al. use gold labels for the class-centroid and supervised
    # attention branches. Phase 2 has no gold set, so the ordered,
    # delexicalized boundary policy supplies explicitly weak pseudo-labels.
    # A stratified holdout measures only agreement with those anchors; it must
    # not be presented as clinical accuracy.
    weak_anchor_labels = queries["anchor_rule_intent"].astype(str).tolist()
    liang_result = fit_liang_hybrid_intent_model(
        query_ids=queries["query_id"].astype(str).tolist(),
        query_texts=queries["intent_text"].astype(str).tolist(),
        query_embeddings=query_embeddings,
        candidates=CANDIDATE_INTENTS,
        candidate_embeddings=candidate_embeddings,
        anchor_labels=weak_anchor_labels,
        stop_words=INTENT_STOP_WORDS,
        config=LiangAdaptationConfig(random_seed=CONFIG.random_seed),
    )
    liang_assignments = liang_result.assignments.reset_index(drop=True)
    for column in liang_assignments.columns:
        if column != "query_id":
            queries[column] = liang_assignments[column].to_numpy()

    cluster_labels, comparison = _fit_cluster_models(query_embeddings)
    queries["cluster_id"] = cluster_labels
    analysis, explanations, profiles = build_analysis_tables(
        queries, structured, tfidf, vectorizer, query_embeddings, prototype_scores, cluster_labels
    )
    manual_review, ambiguity_review, cross_class = build_manual_review(
        queries, query_embeddings, prototype_assignments
    )
    decisions = build_taxonomy_decisions(queries)

    counts = queries["liang_final_intent"].value_counts()
    candidate_table = pd.DataFrame([asdict(intent) for intent in CANDIDATE_INTENTS])
    candidate_table["n_queries"] = candidate_table["intent_name"].map(counts).fillna(0).astype(int)
    candidate_table["support_estimation_method"] = (
        "Liang-style weakly supervised three-branch attention model"
    )
    candidate_table["support_decision"] = np.select(
        [candidate_table["n_queries"].ge(20), candidate_table["n_queries"].ge(10)],
        [">=20: standalone supported", "10-19: retain only after explicit manual exception"],
        default="<10: merge unless clinically essential",
    )
    # This is the actual model-supported final schema. Candidate definitions
    # are hypotheses; they become final only after passing the Phase 2 support
    # threshold. Clinically essential exceptions would require an explicit,
    # documented manual override (none are used in this run).
    final_table = candidate_table[candidate_table["n_queries"].ge(20)].copy()
    derivation = build_seed_to_final_derivation(final_table, liang_assignments, analysis)

    seed_table = pd.DataFrame([asdict(intent) for intent in SEED_INTENTS])
    assignment_export = queries[
        [
            "query_id",
            "query_text",
            "language",
            "intent_text",
            "cluster_id",
            "liang_final_intent",
            "liang_top_probability",
            "liang_second_intent",
            "liang_second_probability",
            "liang_probability_margin",
            "attention_semantic",
            "attention_sentence_tfidf",
            "attention_class_centroid_tfidf",
            "anchor_rule_intent",
            "matched_candidate_intents",
            "n_anchor_matches",
            "used_as_weak_anchor",
        ]
    ].merge(prototype_assignments, on="query_id", how="left")
    model_disagreements = assignment_export[
        assignment_export["liang_final_intent"].ne(assignment_export["anchor_rule_intent"])
    ].copy()
    model_disagreements["review_status"] = "requires Phase 3 adjudication"
    model_disagreements["interpretation"] = (
        "Liang-style model and weak anchor disagree; neither is a gold label"
    )

    paths = {
        "seed_taxonomy": TAXONOMY_DIR / "phase2_seed_taxonomy.csv",
        "prototype_assignments": TAXONOMY_DIR / "phase2_prototype_assignments.csv",
        "liang_assignments": TAXONOMY_DIR / "phase2_liang_hybrid_assignments.csv",
        "liang_branch_metrics": TAXONOMY_DIR / "phase2_liang_branch_metrics.csv",
        "liang_training_history": TAXONOMY_DIR / "phase2_liang_training_history.csv",
        "liang_methodology": TAXONOMY_DIR / "phase2_liang_methodology.json",
        "seed_to_final_derivation": TAXONOMY_DIR / "phase2_seed_to_final_derivation.csv",
        "model_disagreements": TAXONOMY_DIR / "phase2_liang_model_disagreements.csv",
        "cluster_comparison": TAXONOMY_DIR / "phase2_cluster_model_comparison.csv",
        "nlp_analysis": TAXONOMY_DIR / "phase2_nlp_taxonomy_analysis.csv",
        "tfidf_explanations": TAXONOMY_DIR / "phase2_tfidf_explanations.csv",
        "structured_profiles": TAXONOMY_DIR / "phase2_structured_feature_profiles.csv",
        "taxonomy_decisions": TAXONOMY_DIR / "phase2_taxonomy_decisions.csv",
        "manual_review": TAXONOMY_DIR / "phase2_manual_clinical_review.csv",
        "ambiguity_review": TAXONOMY_DIR / "phase2_low_margin_review.csv",
        "cross_class_review": TAXONOMY_DIR / "phase2_cross_class_neighbors.csv",
        "final_taxonomy": TAXONOMY_DIR / "phase2_final_taxonomy.csv",
        "structured_features": TAXONOMY_DIR / "phase2_structured_query_features.csv",
        "report": TAXONOMY_DIR / "phase2_taxonomy_report.md",
        "completion_checklist": TAXONOMY_DIR / "phase2_completion_checklist.md",
        "manifest": TAXONOMY_DIR / "phase2_manifest.json",
    }
    seed_table.to_csv(paths["seed_taxonomy"], index=False)
    assignment_export.to_csv(paths["prototype_assignments"], index=False)
    liang_assignments.to_csv(paths["liang_assignments"], index=False)
    liang_result.branch_metrics.to_csv(paths["liang_branch_metrics"], index=False)
    liang_result.training_history.to_csv(paths["liang_training_history"], index=False)
    paths["liang_methodology"].write_text(
        json.dumps(liang_result.methodology, indent=2), encoding="utf-8"
    )
    derivation.to_csv(paths["seed_to_final_derivation"], index=False)
    model_disagreements.to_csv(paths["model_disagreements"], index=False)
    save_npz(CACHE_DIR / "phase2_liang_tfidf_matrix.npz", liang_result.tfidf_matrix)
    np.save(CACHE_DIR / "phase2_liang_class_centroids.npy", liang_result.class_centroids)
    (CACHE_DIR / "phase2_liang_selected_features.json").write_text(
        json.dumps(liang_result.selected_tfidf_features, indent=2), encoding="utf-8"
    )
    comparison.to_csv(paths["cluster_comparison"], index=False)
    analysis.to_csv(paths["nlp_analysis"], index=False)
    explanations.to_csv(paths["tfidf_explanations"], index=False)
    profiles.to_csv(paths["structured_profiles"], index=False)
    decisions.to_csv(paths["taxonomy_decisions"], index=False)
    manual_review.to_csv(paths["manual_review"], index=False)
    ambiguity_review.to_csv(paths["ambiguity_review"], index=False)
    cross_class.to_csv(paths["cross_class_review"], index=False)
    final_table.to_csv(paths["final_taxonomy"], index=False)
    structured.to_csv(paths["structured_features"], index=False)
    _write_report(
        seed_table,
        analysis,
        final_table,
        decisions,
        comparison,
        prototype_assignments,
        manual_review,
        cross_class,
        derivation,
        liang_result.branch_metrics,
        liang_assignments,
    )
    _write_completion_checklist(
        int(comparison.loc[comparison["selected"], "n_clusters"].iloc[0])
    )
    _write_figures(queries, query_embeddings, cluster_labels)

    manifest = {
        "phase": 2,
        "n_queries": len(queries),
        "semantic_model": CONFIG.semantic_model_name,
        "semantic_model_frozen": True,
        "semantic_inference_device": "cpu",
        "query_embedding_shape": list(query_embeddings.shape),
        "prototype_embedding_shape": list(prototype_embeddings.shape),
        "candidate_intent_embedding_shape": list(candidate_embeddings.shape),
        "tfidf_shape": list(tfidf.shape),
        "structured_feature_count": len(FEATURE_COLUMNS),
        "selected_cluster_method": "agglomerative_cosine",
        "selected_n_clusters": int(comparison.loc[comparison["selected"], "n_clusters"].iloc[0]),
        "selected_silhouette_cosine": round(float(comparison.loc[comparison["selected"], "silhouette_cosine"].iloc[0]), 6),
        "final_taxonomy_size": len(final_table),
        "final_class_distribution": {str(k): int(v) for k, v in counts.sort_index().items()},
        "final_assignments_derived_by": "Liang-style three-branch attention model",
        "weak_anchor_rows": int(liang_assignments["used_as_weak_anchor"].sum()),
        "mean_attention_weights": {
            "semantic": round(float(liang_assignments["attention_semantic"].mean()), 6),
            "sentence_tfidf": round(float(liang_assignments["attention_sentence_tfidf"].mean()), 6),
            "class_centroid_tfidf": round(float(liang_assignments["attention_class_centroid_tfidf"].mean()), 6),
        },
        "liang_validation_metrics_are_gold_accuracy": False,
        "liang_model_anchor_disagreements": len(model_disagreements),
        "manual_review_rows": len(manual_review),
        "low_margin_review_rows": len(ambiguity_review),
        "cross_class_neighbor_pairs": len(cross_class),
        "gold_labels_created": False,
        "random_seed": CONFIG.random_seed,
        "artifacts": {key: str(path.relative_to(path.parents[2])) if path.is_relative_to(path.parents[2]) else str(path) for key, path in paths.items() if key != "manifest"},
    }
    paths["manifest"].write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return paths

In [ ]:
phase2_paths = run_phase2_taxonomy()
phase2_paths

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from src.config import TAXONOMY_DIR

seed_taxonomy = pd.read_csv(TAXONOMY_DIR / 'phase2_seed_taxonomy.csv')
cluster_analysis = pd.read_csv(TAXONOMY_DIR / 'phase2_nlp_taxonomy_analysis.csv')
final_taxonomy = pd.read_csv(TAXONOMY_DIR / 'phase2_final_taxonomy.csv')
taxonomy_decisions = pd.read_csv(TAXONOMY_DIR / 'phase2_taxonomy_decisions.csv')
prototype_assignments = pd.read_csv(TAXONOMY_DIR / 'phase2_prototype_assignments.csv')
liang_assignments = pd.read_csv(TAXONOMY_DIR / 'phase2_liang_hybrid_assignments.csv')
liang_branch_metrics = pd.read_csv(TAXONOMY_DIR / 'phase2_liang_branch_metrics.csv')
seed_to_final = pd.read_csv(TAXONOMY_DIR / 'phase2_seed_to_final_derivation.csv')

display(seed_taxonomy[['literature_intent', 'definition', 'source']])
display(liang_branch_metrics)
display(seed_to_final)
display(cluster_analysis)
display(taxonomy_decisions)
display(final_taxonomy)

### Phase 2 decision

The 500-query sample supports 11 operational classes, all with at least 20 provisional examples: Management / Treatment Selection; Dosing / Administration; Safety / Contraindication; Interaction / Combination; Comparative Treatment Choice; Monitoring / Response / Risk Assessment; Efficacy / Outcomes; Guideline / Evidence Lookup; Treatment Change / Escalation; Prophylaxis / Maintenance; and Mechanism / Background Knowledge.

The broad Pharmacotherapy and Management seeds split into retrieval-distinct needs. Mechanism / Background Knowledge is added because it is coherent and supported. Eight risk-stratification queries merge with monitoring/assessment. Diagnosis, work-up, result interpretation, procedures, and patient education are not supported as learnable classes in this sample. The query set is highly template-patterned, and Liang's original model assumes gold supervision. Phase 3 must therefore adjudicate weak-label boundaries and must not treat model probabilities or assignments as gold labels.

In [ ]:
phase2_distribution = (
    liang_assignments['liang_final_intent']
    .value_counts()
    .rename_axis('intent_name')
    .rename('n_queries')
)
phase2_distribution

## Phase 3: Intent Labeling Strategy

Phase 3 combines three decision channels: ordered bilingual rules, query-to-intent similarity from frozen multilingual embeddings, and unsupervised agglomerative clusters mapped to the intent schema. The cluster mapping is coordinate-optimized to maximize Fleiss' kappa across the three signals, with semantic centroid similarity breaking exact ties. A 60-query pilot emphasizes three-signal disagreements, low margins, rare intents, multi-cue boundaries, all languages, and all clusters. The reviewed reference subset is then expanded to 180 queries with broad coverage.

Kappa here measures algorithmic agreement, not correctness. The prototype and cluster channels share an encoder, the optimization is in-sample, and the reviewed subset is a single model-assisted analyst pass rather than independently clinician-annotated ground truth.

In [ ]:
# Phase 3 orchestration is notebook-owned; reusable labeling helpers remain in src/intent.py.
import json
from pathlib import Path

import pandas as pd

from src.config import OUTPUT_DIR
from src.intent import (
    CLUSTER_MAPPING_PATH, DISAGREEMENT_PATH, ENRICHED_QUERIES_PATH, GOLD_PATH,
    GUIDELINES_PATH, KAPPA_HISTORY_PATH, KAPPA_PATH, LABELED_QUERIES_PATH,
    METRICS_PATH, PHASE3_DIR, PILOT_PATH, PROTOTYPE_ASSIGNMENTS_PATH,
    QUALITY_AUDIT_PATH, REPORT_PATH, ROOT_LABELED_QUERIES_PATH, SIGNAL_AUDIT_PATH,
    build_annotation_frame, build_deliverables, build_guidelines, build_metrics,
    build_report, validate_deliverables,
)


def run_phase3() -> dict[str, Path]:
    """Run Phase 3 and write all auditable annotation artifacts."""
    queries = pd.read_csv(ENRICHED_QUERIES_PATH)
    phase2 = pd.read_csv(PROTOTYPE_ASSIGNMENTS_PATH)
    frame = build_annotation_frame(queries, phase2)
    deliverables = build_deliverables(frame)
    validate_deliverables(deliverables)
    guidelines = build_guidelines()
    metrics = build_metrics(deliverables)

    PHASE3_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    deliverables['pilot'].to_csv(PILOT_PATH, index=False)
    deliverables['gold'].to_csv(GOLD_PATH, index=False)
    deliverables['audit'].to_csv(QUALITY_AUDIT_PATH, index=False)
    deliverables['disagreements'].to_csv(DISAGREEMENT_PATH, index=False)
    deliverables['signal_audit'].to_csv(SIGNAL_AUDIT_PATH, index=False)
    deliverables['cluster_mapping'].to_csv(CLUSTER_MAPPING_PATH, index=False)
    deliverables['kappa_history'].to_csv(KAPPA_HISTORY_PATH, index=False)
    deliverables['labeled'].to_csv(LABELED_QUERIES_PATH, index=False)
    deliverables['labeled'].to_csv(ROOT_LABELED_QUERIES_PATH, index=False)
    guidelines.to_csv(GUIDELINES_PATH, index=False)
    METRICS_PATH.write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False) + '\n', encoding='utf-8'
    )
    kappa_diagnostics = {
        key: value
        for key, value in metrics.items()
        if key.startswith('fleiss_kappa')
        or key in {
            'pairwise_cohen_kappa', 'signal_agreement_distribution',
            'kappa_interpretation',
        }
    }
    KAPPA_PATH.write_text(
        json.dumps(kappa_diagnostics, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    REPORT_PATH.write_text(build_report(metrics), encoding='utf-8')
    return {
        'queries_labeled': LABELED_QUERIES_PATH,
        'queries_labeled_root': ROOT_LABELED_QUERIES_PATH,
        'pilot': PILOT_PATH,
        'reviewed_subset': GOLD_PATH,
        'quality_audit': QUALITY_AUDIT_PATH,
        'adjudicated_disagreements': DISAGREEMENT_PATH,
        'three_signal_assignments': SIGNAL_AUDIT_PATH,
        'cluster_intent_mapping': CLUSTER_MAPPING_PATH,
        'kappa_diagnostics': KAPPA_PATH,
        'kappa_optimization_history': KAPPA_HISTORY_PATH,
        'annotation_guidelines': GUIDELINES_PATH,
        'metrics': METRICS_PATH,
        'report': REPORT_PATH,
    }


phase3_paths = run_phase3()
phase3_paths

In [ ]:
import json
from src.intent import GOLD_PATH, KAPPA_PATH, LABELED_QUERIES_PATH, METRICS_PATH, PILOT_PATH

phase3_labeled = pd.read_csv(LABELED_QUERIES_PATH)
phase3_pilot = pd.read_csv(PILOT_PATH)
phase3_gold = pd.read_csv(GOLD_PATH)
phase3_metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
phase3_kappa = json.loads(KAPPA_PATH.read_text(encoding='utf-8'))

display(phase3_labeled['intent'].value_counts().rename('n_queries').to_frame())
display(phase3_gold.groupby(['intent_subtype', 'language']).size().unstack(fill_value=0))
display(pd.Series(phase3_kappa['pairwise_cohen_kappa'], name='cohen_kappa').to_frame())
display(pd.Series(phase3_kappa['signal_agreement_distribution'], name='n_queries').to_frame())
phase3_metrics

### Phase 3 decision and limitations

All 11 supported intents remain after the pilot; no supplied query needs the `Other / Ambiguous` fallback. Fleiss' kappa across the three algorithmic signals increases from 0.324 to 0.357 after cluster-to-schema optimization. Only 151 queries receive unanimous signals; non-unanimous rows are preserved in the review queue rather than hidden.

When semantic signals disagree with an explicit primary-information-need rule, Phase 3 conservatively retains the auditable rule and lowers confidence. Maximizing kappa can align algorithms on the same error, and the prototype and cluster signals are not statistically independent because they share the frozen encoder. A second qualified clinical annotator and disagreement adjudication remain necessary before calling these labels clinical ground truth.

## Phase 4: Intent Classification Baseline

The baseline uses query text only: word unigram/bigram TF-IDF followed by class-balanced Logistic Regression. It retains all 362 non-three-way weak labels: 151 unanimous cases, 127 two-signal agreements including the rule, and 84 prototype-plus-cluster agreements against the rule. The 138 three-way disagreements are excluded. Five-fold stratified CV measures weak-label replication, while a separate two-fold template-grouped sensitivity analysis measures transfer to unseen query forms where this is feasible.

In [ ]:
# Phase 4 orchestration is notebook-owned; reusable model/evaluation helpers remain in src/.
import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

from src.config import CONFIG, FIGURES_DIR, METRICS_DIR, set_random_seed
from src.intent import INTENT_NAMES, LABELED_QUERIES_PATH, SIGNAL_AUDIT_PATH
from src.intent_baseline import (
    CLASS_CHECKS_PATH, CLUSTER_PURITY_PATH, CONFUSION_FIGURE_PATH,
    CONFUSION_MATRIX_PATH, CV_PREDICTIONS_PATH, ERROR_ANALYSIS_PATH,
    FOLD_METRICS_PATH, GROUPED_SENSITIVITY_PATH,
    METRICS_PATH as PHASE4_METRICS_PATH, MODEL_PATH, NEAREST_PAIRS_PATH,
    PER_CLASS_METRICS_PATH, REPORT_PATH as PHASE4_REPORT_PATH,
    TOP_FEATURES_PATH, TOP_LEVEL_PER_CLASS_PATH, _aggregate_metrics,
    _fit_final_model, _per_class_metrics, _plot_confusion,
    _update_experiment_log, build_report as build_phase4_report,
    build_taxonomy_checks, normalize_template, run_cross_validation,
    run_grouped_sensitivity,
)


def run_phase4() -> dict[str, Path]:
    """Run Phase 4 end to end and persist auditable artifacts."""
    set_random_seed()
    METRICS_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)

    labeled = pd.read_csv(LABELED_QUERIES_PATH)
    signals = pd.read_csv(SIGNAL_AUDIT_PATH)
    frame = labeled[labeled['signal_agreement'].ne('three_way_disagreement')].copy()
    if len(frame) != 362:
        raise AssertionError('Phase 4 expects 362 non-three-way weak labels')
    expected_agreement = {
        'unanimous': 151,
        'two_signal_including_rule': 127,
        'prototype_cluster_against_rule': 84,
    }
    if frame['signal_agreement'].value_counts().to_dict() != expected_agreement:
        raise AssertionError('Weak-label agreement counts do not match Phase 3')
    if set(frame['intent']) != set(INTENT_NAMES):
        raise AssertionError('Weak-label set must cover all supported intent subtypes')
    frame['template'] = frame.apply(normalize_template, axis=1)

    class_checks, cluster_purity, nearest_pairs, sanity = build_taxonomy_checks(
        labeled, frame, signals
    )
    predictions, fold_metrics = run_cross_validation(frame)
    grouped_predictions, grouped_sensitivity = run_grouped_sensitivity(frame)

    subtype_metrics = _aggregate_metrics(
        predictions['true_intent'], predictions['predicted_intent'].to_numpy()
    )
    top_level_metrics = _aggregate_metrics(
        predictions['true_top_level'], predictions['predicted_top_level'].to_numpy()
    )
    template_counts = frame.groupby('intent')['template'].nunique().to_dict()
    fold_support = (
        predictions.groupby(['true_intent', 'fold']).size().groupby(level=0).min().to_dict()
    )
    per_class = _per_class_metrics(
        predictions['true_intent'], predictions['predicted_intent'],
        list(INTENT_NAMES), 'subtype', template_counts, fold_support,
    )
    top_labels = sorted(frame['intent_top_level'].unique())
    top_per_class = _per_class_metrics(
        predictions['true_top_level'], predictions['predicted_top_level'],
        top_labels, 'top_level',
    )
    matrix = confusion_matrix(
        predictions['true_intent'], predictions['predicted_intent'],
        labels=list(INTENT_NAMES),
    )
    matrix_frame = pd.DataFrame(matrix, index=INTENT_NAMES, columns=INTENT_NAMES)
    matrix_frame.index.name = 'reviewed_intent'
    errors = predictions[~predictions['correct_subtype']].sort_values(
        ['probability_margin', 'query_id'], ascending=[False, True]
    )
    model, top_features = _fit_final_model(frame)

    fold_subtype = fold_metrics[fold_metrics['label_level'].eq('subtype')]
    metrics: dict[str, Any] = {
        'experiment': 'T1-B0',
        'model': 'word unigram/bigram TF-IDF + class-balanced Logistic Regression',
        'primary_metric': 'macro_f1',
        'label_source': 'Phase 3 weak labels excluding three-way disagreements',
        'saved_model_fit': 'refit on all 362 retained weak-label rows after cross-validation',
        'n_weak_labeled_queries': int(len(frame)),
        'n_excluded_three_way_disagreements': int(len(labeled) - len(frame)),
        'included_signal_agreement_counts': expected_agreement,
        'out_of_fold_metrics': subtype_metrics,
        'top_level_out_of_fold_metrics': top_level_metrics,
        'template_grouped_sensitivity': grouped_sensitivity,
        'fold_macro_f1_mean': float(fold_subtype['macro_f1'].mean()),
        'fold_macro_f1_std': float(fold_subtype['macro_f1'].std(ddof=1)),
        'taxonomy_sanity': sanity,
        'cv_design': {
            'strategy': 'five-fold stratified row CV over retained weak labels',
            'n_splits': CONFIG.intent_cv_splits,
            'n_template_groups': int(frame['template'].nunique()),
            'largest_template_group': int(frame['template'].value_counts().max()),
            'train_test_template_overlap_each_fold': fold_metrics.loc[
                fold_metrics['label_level'].eq('subtype'), 'template_overlap'
            ].astype(int).tolist(),
            'all_subtypes_in_every_train_and_test_fold': True,
            'random_seed': CONFIG.random_seed,
            'near_duplicate_leakage_note': (
                'near-identical templates cross primary folds because one class has only one '
                'template; use the separate grouped sensitivity result for unseen-template transfer'
            ),
        },
        'hyperparameters': {
            'tfidf_ngram_range': list(CONFIG.intent_tfidf_ngram_range),
            'tfidf_sublinear_tf': True,
            'logistic_regression_c': CONFIG.intent_logreg_c,
            'class_weight': 'balanced',
            'solver': 'liblinear',
            'max_iter': 2000,
        },
        'limitations': [
            'targets are model-assisted weak labels, not independent clinician gold labels',
            'prototype and cluster signals share an encoder and are not independent votes',
            'row-stratified CV allows recurring templates across folds',
            'one retained class has only one independent query template',
            '84 retained targets follow the rule despite prototype and cluster disagreement',
            'Other / Ambiguous has zero examples and is not learnable',
        ],
    }

    class_checks.to_csv(CLASS_CHECKS_PATH, index=False)
    cluster_purity.to_csv(CLUSTER_PURITY_PATH, index=False)
    nearest_pairs.to_csv(NEAREST_PAIRS_PATH, index=False)
    predictions.to_csv(CV_PREDICTIONS_PATH, index=False)
    fold_metrics.to_csv(FOLD_METRICS_PATH, index=False)
    per_class.to_csv(PER_CLASS_METRICS_PATH, index=False)
    top_per_class.to_csv(TOP_LEVEL_PER_CLASS_PATH, index=False)
    matrix_frame.to_csv(CONFUSION_MATRIX_PATH)
    errors.to_csv(ERROR_ANALYSIS_PATH, index=False)
    top_features.to_csv(TOP_FEATURES_PATH, index=False)
    grouped_predictions.to_csv(GROUPED_SENSITIVITY_PATH, index=False)
    PHASE4_METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    PHASE4_REPORT_PATH.write_text(build_phase4_report(metrics, class_checks), encoding='utf-8')
    joblib.dump(model, MODEL_PATH)
    _plot_confusion(matrix, list(INTENT_NAMES))
    _update_experiment_log(subtype_metrics['macro_f1'])

    return {
        'metrics': PHASE4_METRICS_PATH,
        'report': PHASE4_REPORT_PATH,
        'class_checks': CLASS_CHECKS_PATH,
        'cluster_purity': CLUSTER_PURITY_PATH,
        'nearest_class_pairs': NEAREST_PAIRS_PATH,
        'cv_predictions': CV_PREDICTIONS_PATH,
        'fold_metrics': FOLD_METRICS_PATH,
        'per_class_metrics': PER_CLASS_METRICS_PATH,
        'top_level_per_class_metrics': TOP_LEVEL_PER_CLASS_PATH,
        'confusion_matrix': CONFUSION_MATRIX_PATH,
        'confusion_figure': CONFUSION_FIGURE_PATH,
        'errors': ERROR_ANALYSIS_PATH,
        'top_features': TOP_FEATURES_PATH,
        'grouped_sensitivity_predictions': GROUPED_SENSITIVITY_PATH,
        'model': MODEL_PATH,
    }


phase4_paths = run_phase4()
phase4_paths

In [ ]:
from IPython.display import Image, display
from src.intent_baseline import (
    CLASS_CHECKS_PATH, CONFUSION_FIGURE_PATH, FOLD_METRICS_PATH,
    GROUPED_SENSITIVITY_PATH, METRICS_PATH as PHASE4_METRICS_PATH,
    PER_CLASS_METRICS_PATH,
)

phase4_metrics = json.loads(PHASE4_METRICS_PATH.read_text(encoding='utf-8'))
display(pd.DataFrame([phase4_metrics['out_of_fold_metrics']]))
display(pd.read_csv(FOLD_METRICS_PATH))
display(pd.read_csv(PER_CLASS_METRICS_PATH))
display(pd.read_csv(CLASS_CHECKS_PATH))
display(pd.DataFrame([phase4_metrics['template_grouped_sensitivity']]))
display(Image(filename=str(CONFUSION_FIGURE_PATH)))

### Phase 4 decision and limitations

The baseline reaches 0.981 five-fold Macro F1 against the 362 retained weak labels (0.991 macro precision, 0.972 macro recall, 0.983 weighted F1, 0.983 accuracy). Top-level Macro F1 is 0.981. A zero-template-overlap sensitivity analysis reaches only 0.509 Macro F1 across the 10 classes with at least two templates; Monitoring / Response / Risk Assessment has only one retained template and cannot be assessed under an unseen-template split. `Other / Ambiguous` has no examples and cannot be learned.

The primary score measures reproduction of model-assisted weak labels, not clinical accuracy. The 84 prototype-plus-cluster-against-rule cases retain the schema-adjudicated rule target, and the two semantic signals are correlated because they share an encoder. The large row-CV versus grouped-sensitivity gap shows that recurring templates drive substantial performance. Both results are failure-analysis anchors rather than clinical-validity or production-readiness estimates.

## Phase 5: Improve Intent Classification

All Phase 5 supervised variants reuse the exact Phase 4 folds and 362 retained weak labels. The comparison includes final-taxonomy prototype similarity (T1-P0), the frozen Phase 4 lexical baseline (T1-B0), entity indicators/counts (T1-E1), entity plus Phase 1.6 contextual slots (T1-E2), cached frozen multilingual sentence embeddings with Logistic Regression (T1-E3), and early fusion of those embeddings with entity/context features (T1-E4). Model selection uses the zero-template-overlap stress test because row-stratified weak-label replication is already saturated.

In [ ]:
from src.intent_improved import run_phase5

phase5_paths = run_phase5()
phase5_paths

In [ ]:
from src.intent_improved import (
    COMPARISON_PATH as PHASE5_COMPARISON_PATH,
    CONFUSION_FIGURE_PATH as PHASE5_CONFUSION_FIGURE_PATH,
    CONFUSION_PAIRS_PATH as PHASE5_CONFUSION_PAIRS_PATH,
    PER_CLASS_PATH as PHASE5_PER_CLASS_PATH,
    SLICE_METRICS_PATH as PHASE5_SLICE_METRICS_PATH,
)

display(pd.read_csv(PHASE5_COMPARISON_PATH))
display(pd.read_csv(PHASE5_PER_CLASS_PATH).query("protocol == 'template_grouped'"))
display(pd.read_csv(PHASE5_CONFUSION_PAIRS_PATH).head(12))
display(pd.read_csv(PHASE5_SLICE_METRICS_PATH).query("protocol == 'template_grouped' and experiment == 'T1-E3'"))
display(Image(filename=str(PHASE5_CONFUSION_FIGURE_PATH)))

### Phase 5 decision and limitations

T1-E4 is selected: fixed early fusion raises template-held-out Macro F1 to 0.842, an absolute +0.333 over Phase 4's 0.509 and +0.048 over embedding-only T1-E3. Its row-stratified weak-label Macro F1 is 0.970 versus 0.981 for Phase 4. Entity presence/count features alone are effectively neutral at 0.514, contextual slots with TF-IDF reach 0.678, and the embedding/context combination provides the best unseen-template result. The largest remaining confusions are Safety vs Dosing, Prophylaxis vs general Management, Interaction vs Comparison/Management, and Management vs Safety/Prophylaxis.

These are weak-label results, not clinical accuracy. Monitoring remains impossible to evaluate without template leakage, `Other / Ambiguous` has no examples, and the embedding representation contributed to Phase 3 prototype/cluster signals used for evaluation-row filtering. E4 therefore shows promising formulation transfer within this dataset, not an independent clinical benchmark.

## Task 1: Intent Taxonomy and Classifier

Define a doctor-centered intent taxonomy, label queries, train a small-data classifier, and inspect failure cases.

## Task 2: Retrieval and Evaluation

Construct behavior-derived relevance labels, build a BM25 baseline, and evaluate structured, semantic, hybrid, and intent-aware retrieval variants.

## What I Would Build Next

Summarize higher-effort extensions and productionization ideas that are out of scope for the CPU-first baseline.